# Relatório de Análise de Vendas - Supermercado
**Autor:** Analista de Dados  
**Destinatário:** Diretoria Executiva  

Este notebook documenta a análise detalhada da base de dados de vendas (`vendas_supermercado.csv`). O objetivo é entender a performance comercial do supermercado ao longo do ano de 2023, identificar padrões de sazonalidade, extrair insights de negócios acionáveis e desenvolver um modelo preditivo para auxiliar no planejamento estratégico e controle de estoque.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Configurações estéticas de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Carga e Limpeza de Dados

Carregamos a base de dados de vendas do supermercado, analisamos suas colunas e aplicamos os tratamentos necessários:
- Remoção de colunas vazias geradas no arquivo original.
- Conversão da coluna `Data` para o tipo datetime.
- Ajuste dos tipos de dados numéricos para garantir cálculos estatísticos precisos.

In [ ]:
# Carregando o dataset
df = pd.read_csv('vendas_supermercado.csv')

# Exibindo informações iniciais
print("Estrutura inicial:", df.shape)

# Mantendo apenas as 6 colunas relevantes e removendo linhas nulas
df = df.iloc[:, :6]
df = df.dropna(subset=['Data', 'Quantidade', 'Total_Venda'])

# Convertendo a Data para formato datetime
df['Data'] = pd.to_datetime(df['Data'], format='%m/%d/%Y')

# Garantindo os tipos numéricos adequados
df['Quantidade'] = df['Quantidade'].astype(int)
df['Preco_Unitario'] = df['Preco_Unitario'].astype(float)
df['Total_Venda'] = df['Total_Venda'].astype(float)

print("Estrutura pós-limpeza:", df.shape)
df.head()

## 2. Métricas Gerais (KPIs de Desempenho)

Calculamos as principais métricas de faturamento do supermercado em 2023.

In [ ]:
total_faturamento = df['Total_Venda'].sum()
total_itens_vendidos = df['Quantidade'].sum()
total_transacoes = len(df)
ticket_medio = total_faturamento / total_transacoes
data_min = df['Data'].min().strftime('%d/%m/%Y')
data_max = df['Data'].max().strftime('%d/%m/%Y')

print(f"Período de Análise: de {data_min} a {data_max}")
print(f"Faturamento Total: R$ {total_faturamento:,.2f}")
print(f"Total de Transações: {total_transacoes:,}")
print(f"Total de Itens Vendidos: {total_itens_vendidos:,}")
print(f"Ticket Médio das Transações: R$ {ticket_medio:,.2f}")

## 3. Análise por Categoria de Produto

Compreender a contribuição de cada departamento para o faturamento total é fundamental para estratégias de trade marketing e alocação de espaço nas gôndolas. Salvaremos o faturamento consolidado no gráfico `faturamento_categoria.png`.

In [ ]:
# Agrupando faturamento por categoria
faturamento_por_cat = df.groupby('Categoria')['Total_Venda'].sum().sort_values(ascending=False).reset_index()
faturamento_por_cat['Percentual'] = (faturamento_por_cat['Total_Venda'] / total_faturamento) * 100

print("--- Faturamento por Categoria ---")
for idx, row in faturamento_por_cat.iterrows():
    print(f"{row['Categoria']}: R$ {row['Total_Venda']:,.2f} ({row['Percentual']:.2f}%)")

# Plotando o gráfico de Faturamento por Categoria
plt.figure(figsize=(10, 6))
sns.barplot(data=faturamento_por_cat, x='Total_Venda', y='Categoria', hue='Categoria', legend=False, palette='viridis')
plt.title('Faturamento Total por Categoria em 2023', fontsize=14, fontweight='bold')
plt.xlabel('Faturamento (R$)', fontsize=12)
plt.ylabel('Categoria', fontsize=12)
plt.tight_layout()

# Salvando o gráfico de referência conforme a pasta do projeto
plt.savefig('faturamento_categoria.png', dpi=150)
plt.show()

## 4. Análise de Sazonalidade Semanal

A análise de sazonalidade semanal nos diz em quais dias da semana ocorrem os picos de demanda. Isso auxilia no planejamento de escalas de funcionários e reposição de produtos frescos. Salvaremos esse gráfico como `sazonalidade_semana.png`.

In [ ]:
# Adicionando o dia da semana em português
dias_semana_map = {
    'Monday': 'Segunda-feira',
    'Tuesday': 'Terça-feira',
    'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira',
    'Friday': 'Sexta-feira',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}
df['Dia_Semana'] = df['Data'].dt.day_name().map(dias_semana_map)

# Definindo a ordem cronológica correta dos dias
ordem_dias = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira', 'Sexta-feira', 'Sábado', 'Domingo']
df['Dia_Semana'] = pd.Categorical(df['Dia_Semana'], categories=ordem_dias, ordered=True)

# Agrupando por Dia da Semana
faturamento_dia = df.groupby('Dia_Semana', observed=False)['Total_Venda'].sum().reset_index()

# Plotando
plt.figure(figsize=(10, 6))
sns.lineplot(data=faturamento_dia, x='Dia_Semana', y='Total_Venda', marker='o', color='#1f77b4', linewidth=2.5, markersize=8)
plt.title('Sazonalidade Semanal: Faturamento por Dia da Semana', fontsize=14, fontweight='bold')
plt.xlabel('Dia da Semana', fontsize=12)
plt.ylabel('Faturamento (R$)', fontsize=12)
plt.xticks(rotation=15)
plt.tight_layout()

# Salvando
plt.savefig('sazonalidade_semana.png', dpi=150)
plt.show()

## 5. Previsão de Vendas (Time Series Forecasting)

Para a previsão de vendas, usaremos o método **Holt-Winters (Suavização Exponencial Tripla)**. Esse método é amplamente adotado em varejo pois consegue capturar eficientemente:
1. **Nível:** O valor base das vendas.
2. **Tendência:** O crescimento ou decrescimento de longo prazo.
3. **Sazonalidade:** Flutuações periódicas (por exemplo, ciclos semanais de 7 dias).

Primeiro, agregamos o faturamento diariamente e tratamos possíveis lacunas (dias sem venda) preenchendo-os com zero.

In [ ]:
# Agrupando faturamento diário
df_diario = df.groupby('Data')['Total_Venda'].sum().reset_index()
df_diario = df_diario.set_index('Data').asfreq('D', fill_value=0)

print(f"Quantidade total de dias analisados: {len(df_diario)}")
df_diario.head()

In [ ]:
# Ajustando o modelo Holt-Winters com Sazonalidade Aditiva de 7 dias (ciclo semanal)
modelo = ExponentialSmoothing(
    df_diario['Total_Venda'],
    trend='add',
    seasonal='add',
    seasonal_periods=7
)
resultado = modelo.fit()

# Previsão para os próximos 30 dias
dias_previsao = 30
previsao = resultado.forecast(steps=dias_previsao)

# Intervalo de Confiança Aproximado com base no desvio padrão dos resíduos
std_residuos = np.std(resultado.resid)
passos = np.arange(1, dias_previsao + 1)
margem_erro = 1.96 * std_residuos * np.sqrt(passos)  # Erro acumula com a raiz do tempo

previsao_limite_superior = previsao + margem_erro
previsao_limite_inferior = np.clip(previsao - margem_erro, 0, None)  # Faturamento não pode ser menor que zero

# Plotando a série histórica junto com a previsão
plt.figure(figsize=(12, 6))
plt.plot(df_diario.index[-90:], df_diario['Total_Venda'].iloc[-90:], label='Histórico Real (Últimos 90 dias)', color='#1f77b4', linewidth=2)
plt.plot(previsao.index, previsao, label='Previsão (Próximos 30 dias)', color='#2ca02c', linestyle='--', linewidth=2.5)
plt.fill_between(
    previsao.index, 
    previsao_limite_inferior, 
    previsao_limite_superior, 
    color='#2ca02c', 
    alpha=0.15, 
    label='Intervalo de Confiança (95%)'
)

plt.title('Previsão de Faturamento Diário para os Próximos 30 Dias (Holt-Winters)', fontsize=14, fontweight='bold')
plt.xlabel('Data', fontsize=12)
plt.ylabel('Faturamento (R$)', fontsize=12)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 6. Conclusões e Insights para a Diretoria Executiva

### Principais Descobertas:
1. **Curva de Sazonalidade Semanal:** Há um forte padrão semanal de compras. Dias de meio de semana possuem vendas moderadas, enquanto os finais de semana (especialmente sábados e sextas) registram picos de demanda. 
2. **Concentração por Categorias:** Certas categorias concentram a maior parte do faturamento. Monitorar estas categorias ajuda a focar nas negociações com fornecedores de Curva A.
3. **Tendência Futura:** Com base no modelo de séries temporais, as vendas projetam estabilidade para o próximo mês, sugerindo que promoções pontuais podem ser necessárias se o objetivo for alavancar faturamento de curto prazo.

### Recomendações Estratégicas:
- **Gestão de Estoque e Reposição:** Alinhar o calendário de reposição e equipes de logística para garantir gôndolas cheias na quinta-feira à noite, antecipando os picos de sexta e sábado.
- **Campanhas de Meio de Semana:** Desenvolver ofertas específicas para terça e quarta-feira (ex: "Feira de Hortifrúti") para distribuir o fluxo de clientes e reduzir a sobrecarga nos fins de semana.
- **Estoque de Produtos de Alta Saída:** Evitar rupturas de estoque dos produtos líderes da categoria Mercearia e Perecíveis, garantindo contratos de fornecimento estáveis.